# Ride Fare Prediction using MLOps Pipeline

## Objective
To build an end-to-end MLOps pipeline that includes:
- Data preprocessing
- Data versioning (DVC)
- Model training
- Experiment tracking (MLflow)
- Pipeline automation
- CI/CD
- Deployment (FastAPI)

In [19]:
import pandas as pd

df = pd.read_csv("data/uber.csv")
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [22]:
df.shape

(200000, 9)

## Data Preprocessing

The raw Uber dataset was cleaned using a preprocessing script:
- Removed missing values
- Extracted hour from datetime
- Calculated approximate distance
- Reduced dataset size for efficient processing

In [3]:
# Data versioning using DVC
# Command used:
# !dvc add data/ride_data.csv

In [25]:
import pandas as pd

df = pd.read_csv("data/ride_data.csv")
df.head()

,distance_km,hour,fare
0,3.286488,1,6.5
1,1.586745,15,4.1
2,3.900318,17,12.0
3,4.127979,23,7.5
4,1.222110,8,4.1


In [26]:
df.describe()

,distance_km,hour,fare
count,2000.000000,2000.000000,2000.000000
mean,36.733183,13.693000,11.443860
std,635.839900,6.390162,10.920254
min,0.000000,0.000000,2.500000
25%,1.731739,9.000000,6.000000
50%,3.081693,14.000000,8.500000
75%,5.453929,19.000000,12.500000
max,12738.291179,23.000000,196.000000


In [27]:
df.isnull().sum()

distance_km    0
hour           0
fare           0
dtype: int64

## Experiment Tracking (MLflow)

MLflow is used to track:
- Model performance (R2 score)
- Parameters
- Model artifacts

Runs can be viewed using:
mlflow ui

In [5]:
!dvc repro

'data\ride_data.csv.dvc' didn't change, skipping
Stage 'train' is cached - skipping run, checking out outputs
Use `dvc push` to send your updates to remote storage.


## Data Versioning (DVC)

We track dataset versions using DVC.

In [28]:
!dvc status

train:
	changed outs:
		modified:           model\model.pkl


MODEL TRAINING

In [29]:
!python src/train.py

Model trained. R2 Score: 0.5598575470870559


c:\Users\nihar\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/19 16:50:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 16:50:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MODEL PROOF

In [30]:
import os
os.listdir("model")

['model.pkl']

## Experiment Tracking (MLflow)

MLflow tracks:
- Metrics (R2 score)
- Parameters
- Model artifacts

View using:
mlflow ui

In [31]:
!dvc repro

'data\ride_data.csv.dvc' didn't change, skipping
Stage 'train' is cached - skipping run, checking out outputs
Use `dvc push` to send your updates to remote storage.


API TEST PROOF

In [ ]:
import requests

url = "http://127.0.0.1:8000/predict"

data = {
    "distance_km": 5,
    "hour": 18
}

response = requests.post(url, json=data)
response.json()

LOGGING PROOF

In [ ]:
with open("logs/logs.txt", "r") as f:
    print(f.read())

## CI/CD

We implemented CI/CD using GitHub Actions.
Pipeline runs automatically on push.